# NumPy Matrix Ops Refresher — Demos + Immediate Practice (Solution Key)

**Duration:** ~60 minutes  
**Audience:** Participants building LLMs with Python who need a fast but solid refresher on core NumPy operations used across ML.

**How to use this notebook**
- Each topic has a short *Demo* cell followed immediately by a *Practice* cell.  
- Run the demo, then complete the practice.  
- Topics build from basic array mechanics toward small ML‑style tasks.

**What you'll cover**
1. Arrays & shapes
2. Indexing, slicing, masking
3. Broadcasting
4. Vector & matrix multiplication (`dot`, `@`, `einsum`)
5. Norms and normalization / standardization
6. Axis‑wise reductions and softmax
7. One‑hot encoding & “embedding lookup”
8. Mini linear regression with the normal equation
9. (Bonus) Numerical stability: log‑sum‑exp

> Tip: Use `arr.shape`, `arr.ndim`, and `np.newaxis` frequently; they’re everywhere in deep learning code.

In [1]:
# Setup
import numpy as np
np.set_printoptions(suppress=True, precision=4)
rng = np.random.default_rng(42)  # deterministic randomness
print("NumPy version:", np.__version__)

NumPy version: 2.4.1


## 1) Arrays & shapes
Understand how vectors (1D), matrices (2D), and batched tensors (3D+) appear in ML code.

In [2]:
# Demo: create arrays and inspect shapes
a = np.array([1., 2., 3.])         # vector
B = np.array([[1., 2., 3.],        # matrix (2x3)
              [4., 5., 6.]])
C = rng.normal(size=(4, 3, 2))     # batch of matrices (batch=4)

print("a.shape:", a.shape, "| a.ndim:", a.ndim)
print("B.shape:", B.shape, "| B.ndim:", B.ndim)
print("C.shape:", C.shape, "| C.ndim:", C.ndim)

# Reshape and ravel: common in preparing features
x = rng.normal(size=12)
X = x.reshape(3, 4)  # 3 rows, 4 cols
print("X:\n", X)
print("X.ravel():", X.ravel())

a.shape: (3,) | a.ndim: 1
B.shape: (2, 3) | B.ndim: 2
C.shape: (4, 3, 2) | C.ndim: 3
X:
 [[-0.4283 -0.3521  0.5323  0.3654]
 [ 0.4127  0.4308  2.1416 -0.4064]
 [-0.5122 -0.8138  0.616   1.129 ]]
X.ravel(): [-0.4283 -0.3521  0.5323  0.3654  0.4127  0.4308  2.1416 -0.4064 -0.5122
 -0.8138  0.616   1.129 ]


In [3]:
# Practice (Solution)
v = rng.normal(size=24)
T = v.reshape(3, 4, 2)
print("T.shape should be (3, 4, 2) ->", T.shape)

T.shape should be (3, 4, 2) -> (3, 4, 2)


## 2) Indexing, slicing, masking
Slicing along axes and boolean masks are essential for data filtering and minibatching.

In [4]:
# Demo: slicing rows/cols and boolean masks
X = rng.normal(loc=0.0, scale=1.0, size=(5, 4))  # 5 samples, 4 features
print("X:\n", X)

first_two_rows = X[:2]
last_column = X[:, -1]
big_mask = X[:, 0] > 0.0  # samples where feature0 > 0
X_big = X[big_mask]

print("first_two_rows shape:", first_two_rows.shape)
print("last_column shape:", last_column.shape)
print("mask:", big_mask)
print("X_big shape:", X_big.shape)

X:
 [[-1.6829 -0.3349  0.1628  0.5862]
 [ 0.7112  0.7933 -0.3487 -0.4624]
 [ 0.858  -0.1913 -1.2757 -1.1333]
 [-0.9195  0.4972  0.1424  0.6905]
 [-0.4273  0.1585  0.6256 -0.3093]]
first_two_rows shape: (2, 4)
last_column shape: (5,)
mask: [False  True  True False False]
X_big shape: (2, 4)


In [5]:
# Practice (Solution)
X = rng.normal(size=(8, 5))
row_mask = X.mean(axis=1) > 0
X_sel = X[row_mask][:, [1, 3]]
print("Row mask:", row_mask)
print("X_sel shape should be (num_selected, 2) ->", X_sel.shape)

Row mask: [False  True False False  True False False  True]
X_sel shape should be (num_selected, 2) -> (3, 2)


## 3) Broadcasting
Align shapes without explicit loops. Core to normalization and batched math.

In [6]:
# Demo: feature-wise centering using broadcasting
X = rng.normal(size=(6, 3))  # 6 samples, 3 features
mu = X.mean(axis=0)          # (3,)
centered = X - mu            # (6,3) - (3,) -> (6,3)
print("mu:", mu)
print("centered mean (approx 0):", centered.mean(axis=0))

mu: [ 0.082  -0.0788 -0.3454]
centered mean (approx 0): [ 0.  0. -0.]


In [7]:
# Practice (Solution)
X = rng.normal(size=(10, 4))
mu = X.mean(axis=0)
sigma = X.std(axis=0, ddof=0)  # population std
Z = (X - mu) / sigma
print("Z mean (approx 0):", Z.mean(axis=0))
print("Z std  (approx 1):", Z.std(axis=0))

Z mean (approx 0): [-0. -0. -0.  0.]
Z std  (approx 1): [1. 1. 1. 1.]


## 4) Vector & matrix multiplication (`dot`, `@`, `einsum`)
Compute linear combinations and batched projections.

In [8]:
# Demo: linear model y = Xw + b
X = rng.normal(size=(5, 3))
w = rng.normal(size=(3,))   # weights
b = 0.5
y = X @ w + b               # same as np.dot(X, w) + b
print("y:", y)

# einsum: explicit dimension mapping (useful for clarity/perf)
y2 = np.einsum('ij,j->i', X, w) + b
print("y2 (should match y):", y2)

y: [-1.1583  0.2857 -0.7375 -1.252   3.5606]
y2 (should match y): [-1.1583  0.2857 -0.7375 -1.252   3.5606]


In [9]:
# Practice (Solution)
X = rng.normal(size=(7, 4))
W = rng.normal(size=(4, 2))
b = rng.normal(size=(2,))
Y = X @ W + b  # broadcasting adds b to each row
print("Y.shape should be (7, 2) ->", Y.shape)

Y.shape should be (7, 2) -> (7, 2)


## 5) Norms and normalization / standardization
L2 norms and unit‑length feature vectors are common in similarity tasks.

In [10]:
# Demo: L2-normalize rows (common before cosine similarity)
X = rng.normal(size=(4, 5))
l2 = np.linalg.norm(X, ord=2, axis=1, keepdims=True)  # (4,1)
X_unit = X / (l2 + 1e-12)  # avoid divide-by-zero
row_norms = np.linalg.norm(X_unit, axis=1)
print("Row norms after normalization (approx 1):", row_norms)

Row norms after normalization (approx 1): [1. 1. 1. 1.]


In [11]:
# Practice (Solution)
X = rng.normal(loc=3.0, scale=2.0, size=(12, 6))
mu = X.mean(axis=0)
sd = X.std(axis=0)
Z = (X - mu) / sd
print("mean ~0:", Z.mean(axis=0))
print("std  ~1:", Z.std(axis=0))

mean ~0: [ 0.  0.  0.  0. -0.  0.]
std  ~1: [1. 1. 1. 1. 1. 1.]


## 6) Axis‑wise reductions and softmax
Summations/means across axes and a numerically stable softmax.

In [12]:
# Demo: stable softmax over classes (axis=1)
def softmax(logits, axis=-1):
    z = logits - logits.max(axis=axis, keepdims=True)  # stability shift
    exp = np.exp(z)
    return exp / exp.sum(axis=axis, keepdims=True)

logits = rng.normal(size=(3, 5))  # 3 samples, 5 classes
probs = softmax(logits, axis=1)
print("Row sums (should be 1):", probs.sum(axis=1))
print("Argmax classes:", probs.argmax(axis=1))

Row sums (should be 1): [1. 1. 1.]
Argmax classes: [4 0 2]


In [13]:
# Practice (Solution)
L = rng.normal(size=(6, 4))
def col_softmax(A):
    z = A - A.max(axis=0, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=0, keepdims=True)
P = col_softmax(L)
print("Col sums (should be ~1):", P.sum(axis=0))

Col sums (should be ~1): [1. 1. 1. 1.]


## 7) One‑hot encoding & “embedding lookup”
Map integer token IDs to rows of an embedding matrix; core to NLP & LLMs.

In [14]:
# Demo: one-hot and lookup
vocab_size = 8
embed_dim = 4
token_ids = np.array([3, 1, 0, 6])   # length 4 sequence
E = rng.normal(size=(vocab_size, embed_dim))

# One-hot via eye:
one_hot = np.eye(vocab_size)[token_ids]   # (4, vocab_size)
# Equivalent "embedding lookup" via gather:
embeds = E[token_ids]                     # (4, embed_dim)

print("one_hot shape:", one_hot.shape)
print("embeds shape:", embeds.shape)
# Check equivalence: one_hot @ E == embeds
print("Equivalent lookup:", np.allclose(one_hot @ E, embeds))

one_hot shape: (4, 8)
embeds shape: (4, 4)
Equivalent lookup: True


In [15]:
# Practice (Solution)
batch_token_ids = np.array([[0, 1, 1, 2],
                            [3, 3, 4, 5],
                            [6, 7, 7, 7]])
E = rng.normal(size=(10, 6))  # vocab=10, dim=6
seq_embeds = E[batch_token_ids]          # (3, 4, 6)
avg_embeds = seq_embeds.mean(axis=1)     # (3, 6)
print("avg_embeds shape should be (3, 6) ->", avg_embeds.shape)

avg_embeds shape should be (3, 6) -> (3, 6)


## 8) Mini linear regression with the normal equation
Fit \(w, b\) by closed form. Useful to connect linear algebra with ML loss minimization.

In [16]:
# Demo: generate synthetic linear data and solve (X^T X)^{-1} X^T y
n, d = 100, 3
X = rng.normal(size=(n, d))
true_w = rng.normal(size=d)
true_b = -0.7
noise = rng.normal(scale=0.1, size=n)
y = X @ true_w + true_b + noise

# Add bias column of 1s
X_aug = np.c_[X, np.ones(n)]
theta_hat = np.linalg.pinv(X_aug) @ y   # (d+1,)
w_hat, b_hat = theta_hat[:-1], theta_hat[-1]

mse = np.mean((X @ w_hat + b_hat - y)**2)
print("true_w:", true_w, "true_b:", true_b)
print("w_hat :", w_hat,  "b_hat :", b_hat)
print("train MSE:", mse)

true_w: [0.855  0.6396 0.4425] true_b: -0.7
w_hat : [0.8503 0.6427 0.4278] b_hat : -0.6941419171677803
train MSE: 0.009769067079045554


In [17]:
# Practice (Solution)
X_new = rng.normal(size=(50, 3))
noise = rng.normal(scale=0.1, size=50)
y_true = X_new @ true_w + true_b + noise

y_pred = X_new @ w_hat + b_hat
mse_new = np.mean((y_pred - y_true)**2)
print("MSE on new data (should be close to noise variance 0.01):", mse_new)

MSE on new data (should be close to noise variance 0.01): 0.009305469588454553


## 9) (Bonus) Numerical stability — log‑sum‑exp
Avoid overflow/underflow when turning logits into probabilities.

In [18]:
# Demo + Practice: implement logsumexp that is stable
def logsumexp(x, axis=None, keepdims=False):
    m = x.max(axis=axis, keepdims=True)
    y = np.log(np.exp(x - m).sum(axis=axis, keepdims=True)) + m
    if not keepdims:
        y = np.squeeze(y, axis=axis)
    return y

big = np.array([1000.0, 1001.0, 1002.0])
naive = np.log(np.exp(big).sum())
stable = logsumexp(big)
print("Naive (inf expected):", naive)
print("Stable:", stable)

Naive (inf expected): inf
Stable: 1002.4076059644444


/tmp/ipykernel_549/976895895.py:10: RuntimeWarning: overflow encountered in exp
  naive = np.log(np.exp(big).sum())


---

## Wrap‑up
You reviewed shapes, slicing/masking, broadcasting, matrix multiplication, normalization, reductions/softmax, one‑hot/embeddings, and a tiny linear regression—all pillars for reading & writing ML/LLM NumPy code.